## C-Drama RAG: Embeddings
We embed each drama's tags + synopsis into a 1536-dimensional vector
using OpenAI's `text-embedding-3-large` model. These vectors power semantic search.

**Why omit genres?** Genres are already applied as a hard SQL filter in `match_documents`,
so baking them into the embedding double-weights them. Tags capture specific narrative
tropes (e.g. `fake to real lovers`, `amnesia`) that are the real semantic signal;
synopsis provides the story-level meaning.

### Setup

In [1]:
import pandas as pd
import os
import time
from pathlib import Path
from openai import OpenAI
from src.env import load_secrets
# load environment variables

load_secrets()

client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

INPUT_PATH = Path("../data/cleaned/dramas_clean.jsonl")
OUTPUT_PATH = Path("../data/cleaned/dramas_with_vectors.parquet")

EMBEDDING_MODEL = "text-embedding-3-large"
DIMENSIONS = 1536
BATCH_SIZE = 200
COST_PER_1M_TOKENS = 0.13

### Load clean data

In [2]:
df = pd.read_json(INPUT_PATH, lines=True)
print(f"Loaded {len(df)} dramas")

Loaded 1421 dramas


### Build the embedding context string

Genres are excluded — handled by the SQL `filter_genres` parameter in `match_documents`.
Title and year intentionally excluded — those are for SQL filters, not semantic meaning.

In [3]:
tags_str = df["tags"].apply(", ".join)

df["context"] = (
    "Tags: " + tags_str + "\n" +
    "Synopsis: " + df["synopsis"]
)

print("Sample context:")
print(df["context"].iloc[0][:500])  # .iloc[0] = first row by integer position

Sample context:
Tags: fake to real lovers, physically strong female lead, female lead saves male lead, marriage of convenience, general male lead, righteous male lead, skilled fighter female lead, hidden identity, physically strong male lead, fake marriage
Synopsis: It follows Fan Chang Yu, a butcher’s daughter, and Xie Zheng, a fallen noble seeking revenge. Their fake marriage turns into true love, but war tears them apart. Determined, Fan Chang Yu wields her butcher’s knife on the battlefield, searching for j


### Cost estimate

In [4]:
# Rough cost estimate (4 chars ≈ 1 token)
total_chars = df["context"].str.len().sum()
est_tokens = total_chars / 4
est_cost = (est_tokens / 1_000_000) * COST_PER_1M_TOKENS

print(f"Total characters: {total_chars:,}")
print(f"Estimated tokens: {est_tokens:,.0f}")
print(f"Estimated cost: ${est_cost:.4f}")

Total characters: 1,124,517
Estimated tokens: 281,129
Estimated cost: $0.0365


### Embedding function with batching and retry

In [5]:
def get_embeddings_in_batches(texts: list[str], batch_size: int = BATCH_SIZE) -> list:
    """Send texts to OpenAI in batches and return all embeddings.
    
    Batching prevents timeouts and rate limit errors.
    Retry logic handles transient API failures.
    """
    all_embeddings = []
    total_batches = (len(texts) + batch_size - 1) // batch_size

    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        batch_num = i // batch_size + 1
        print(f"Batch {batch_num}/{total_batches} ({len(batch)} texts)...", end=" ")

        for attempt in range(3):
            try:
                response = client.embeddings.create(
                    input=batch,
                    model=EMBEDDING_MODEL,
                    dimensions=DIMENSIONS
                )
                batch_embeddings = [item.embedding for item in response.data]
                all_embeddings.extend(batch_embeddings)
                print("✓")
                break

            except Exception as e:
                print(f"\n  Attempt {attempt + 1} failed: {e}")
                if attempt < 2:
                    time.sleep(5)
                else:
                    raise RuntimeError(f"Batch {batch_num} failed after 3 attempts") from e

    return all_embeddings

### Generate embeddings

In [6]:
texts = df["context"].tolist()

print("Starting embedding generation...")
embeddings = get_embeddings_in_batches(texts)
print(f"\nGot {len(embeddings)} embeddings")
print(f"Dimensions: {len(embeddings[0])}")

Starting embedding generation...
Batch 1/8 (200 texts)... ✓
Batch 2/8 (200 texts)... ✓
Batch 3/8 (200 texts)... ✓
Batch 4/8 (200 texts)... ✓
Batch 5/8 (200 texts)... ✓
Batch 6/8 (200 texts)... ✓
Batch 7/8 (200 texts)... ✓
Batch 8/8 (21 texts)... ✓

Got 1421 embeddings
Dimensions: 1536


### Attach embeddings to DataFrame

Pandas will store each embedding (a list of 1536 floats) as a Python object in the column.

In [7]:
df["embedding"] = embeddings

final_cols = [
    "mdl_id", "mdl_url", "title", "native_title",
    "synopsis", "episodes", "year", "genres", "tags",
    "mdl_score", "watchers", "embedding"
]
df_final = df[final_cols]

print(f"Final shape: {df_final.shape}")
df_final.head(3)

Final shape: (1421, 12)


,mdl_id,mdl_url,title,native_title,synopsis,episodes,year,genres,tags,mdl_score,watchers,embedding
0,760409,https://mydramalist.com/760409-zhu-yu,Pursuit of Jade,逐玉,"It follows Fan Chang Yu, a butcher’s daughter,...",40,2026,"[historical, mystery, romance, war]","[fake to real lovers, physically strong female...",9.1,40464,"[-0.00739288330078125, -0.0269317626953125, -0..."
1,53505,https://mydramalist.com/53505-the-untamed-spec...,The Untamed Special Edition,陈情令特別剪辑版,A 20-episode cut of The Untamed made for bette...,20,2019,"[mystery, wuxia, fantasy]","[bromance, smart male lead, xianxia, calm male...",9.0,39873,"[-0.0003440380096435547, -0.00318145751953125,..."
2,9025,https://mydramalist.com/9025-nirvana-in-fire,Nirvana in Fire,琅琊榜,"In sixth-century China, the Emperor of Great L...",54,2015,"[military, historical, drama, political]","[power struggle, smart male lead, scheme, deat...",9.0,60460,"[-0.002349853515625, 0.008544921875, -0.010848..."


### Save to parquet

Parquet is a binary columnar format — much faster to read than JSON, and smaller on disk.
It also preserves types correctly (including the embedding lists).

In [8]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

df_final.to_parquet(OUTPUT_PATH, index=False)  

print(f"Saved to {OUTPUT_PATH}")

# Quick verification — reload and check
check = pd.read_parquet(OUTPUT_PATH)
print(f"Verified: {len(check)} dramas, {len(check.columns)} columns")
print(f"Embedding dims: {len(check['embedding'].iloc[0])}")

Saved to ..\data\cleaned\dramas_with_vectors.parquet
Verified: 1421 dramas, 12 columns
Embedding dims: 1536
